In [1]:
import phoenix as px
from phoenix.otel import register
from openinference.instrumentation.langchain import LangChainInstrumentor

# 1. Запускаем сам сервер Phoenix
session = px.launch_app()

# 2. Регистрируем провайдер трассировки (OpenTelemetry)
# Он будет перехватывать данные и отправлять их в локальный Phoenix
tracer_provider = register()

# 3. Включаем "прослушку" именно для LangChain
LangChainInstrumentor().instrument(tracer_provider=tracer_provider)

print(f"Phoenix готов! Дашборд тут: {session.url}")

/Users/artemzmailov/Desktop/kitoboy-PII/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/artemzmailov/.local/share/uv/python/cpython-3.11.15-macos-aarch64-none/lib/python3.11/contextlib.py:144: SAWarning: Skipped unsupported reflection of expression-based index ix_cumulative_llm_token_count_total
  next(self.gen)
/Users/artemzmailov/.local/share/uv/python/cpython-3.11.15-macos-aarch64-none/lib/python3.11/contextlib.py:144: SAWarning: Skipped unsupported reflection of expression-based index ix_latency
  next(self.gen)


🌍 To view the Phoenix app in your browser, visit http://localhost:6006/
📖 For more information on how to use Phoenix, check out https://arize.com/docs/phoenix
🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: default
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: localhost:4317
|  Transport: gRPC
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.

Phoenix готов! Дашборд тут: http://localhost:6006/


In [5]:
import os
from dotenv import load_dotenv

load_dotenv()

# Теперь к ним можно обращаться через стандартный модуль os
api_key = os.getenv("MISTRAL_API_KEY")
print(api_key) 

Lld8KeTo2clg7lurqzk9G0XZlCuGBdhx


In [16]:
import yaml
import re
from typing import List, Set, TypedDict, Annotated, Literal
from pydantic import BaseModel, Field
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, END

# Модель для structured_output (как ты и писал)
class Entities(BaseModel):
    items: List[str] = Field(description="Список сгенерированных сущностей")

class SeedBucket(BaseModel):
    mode: Literal["digits", "letters", "alnum"] = Field(description="Тип seed-строк: digits, letters или alnum")
    count: int = Field(description="Сколько seed-строк такого типа нужно сгенерировать для одного батча")
    length: int = Field(description="Длина одной seed-строки такого типа")
    purpose: str = Field(description="Для каких частей сущности использовать этот bucket")

class SeedPlan(BaseModel):
    buckets: List[SeedBucket] = Field(description="Набор seed-bucket-ов для разных типов частей сущности")
    reason: str = Field(description="Коротко почему выбраны такие bucket-ы")

# Состояние нашего подграфа
class SubAgentState(TypedDict):
    entity_key: str          # Ключ из YAML (например, 'PASSPORT_RF')
    target_count: int        # Сколько всего нужно уникальных штук
    unique_samples: Set[str] # Наше множество (авто-дедупликация)
    last_batch: List[str]    # Последний выхлоп модели (для логов/проверки)
    iterations: int             # Счетчик итераций

In [17]:
from langchain_core.rate_limiters import InMemoryRateLimiter
from langchain_mistralai import ChatMistralAI

rate_limiter = InMemoryRateLimiter(
    requests_per_second=0.3,   
    check_every_n_seconds=0.1,
    max_bucket_size=1,
)

In [18]:
from langchain_mistralai import ChatMistralAI
from langchain_core.tools import tool
import random
import string

_rng = random.SystemRandom()

@tool
def generate_seed(length: int, mode: str = "digits", count: int = 1) -> list[str]:
    """Генерирует уникальные случайные seed-строки.
    mode: digits | letters | alnum
    """
    if length <= 0 or count <= 0:
        raise ValueError("length and count must be > 0")

    mode = mode.lower()
    if mode == "digits":
        alphabet = string.digits
    elif mode == "letters":
        alphabet = string.ascii_lowercase
    elif mode == "alnum":
        alphabet = string.ascii_lowercase + string.digits
    else:
        raise ValueError("mode must be one of: digits, letters, alnum")

    out = set()
    while len(out) < count:
        out.add("".join(_rng.choice(alphabet) for _ in range(length)))
    return list(out)


# Инициализация модели
llm = ChatMistralAI(model="mistral-medium-latest", #mistral-small-2506
                    temperature=0.7,
                    timeout = 60,
                    rate_limiter=rate_limiter,
                    max_concurrent_requests=1,
                    max_retries=3)
seed_planner_llm = llm.with_structured_output(SeedPlan)
structured_llm = llm.with_structured_output(Entities)

#llm = ChatOllama(model="qwen3.5:9b", reasoning = False, format="json", temperature=0.7)
# seed_planner_llm = llm.with_structured_output(SeedPlan)
# structured_llm = llm.with_structured_output(Entities)
# 
# Загрузка промптов (предположим, файл лежит рядом)
with open('../../configs/prompts.yaml', 'r', encoding='utf-8') as f:
    PROMPTS_CONFIG = yaml.safe_load(f)


def generate_batch_node(state: SubAgentState):
    entity_key = state['entity_key']
    config = PROMPTS_CONFIG.get('ENTITIES', {}).get(entity_key)

    if config is None:
        raise ValueError(f"Сущность {entity_key} не найдена в секции ENTITIES в prompts.yaml")

    # Считаем, сколько еще не хватает до цели
    needed = state['target_count'] - len(state['unique_samples'])
    batch_size = min(needed, 10) # Генерим не больше 10 за раз для качества

    # Берем системный промпт и подставляем N
    sys_prompt = config['system_prompt'].format(N=batch_size)

    seed_plan_messages = [
        SystemMessage(content=(
            "Ты подбираешь seed-bucket-ы для генератора случайных строк. "
            "Seed нужен только как вспомогательный источник сырых символов, а не как формат будущих примеров. "
            "Стиль, структура, шум, разделители и обрамление задаются промптом сущности. "
            "Верни только структуру SeedPlan."
        )),
        HumanMessage(content=(
            f"Сущность: {entity_key}\n\n"
            f"Промпт сущности:\n{sys_prompt}\n\n"
            f"Нужно подготовить seed-bucket-ы для батча из {batch_size} примеров.\n"
            "Можно вернуть один или несколько bucket-ов. Суммарный count по bucket-ам обычно должен быть около размера батча, но может быть меньше, если seed почти не нужен.\n"
            "mode='digits' — для частей, которые по промпту выглядят числовыми: id, номера, цифровые хвосты, коды.\n"
            "mode='letters' — для чисто буквенных частей.\n"
            "mode='alnum' — для username/nickname/login-like частей, где по промпту допустимы буквы и цифры.\n"
            "length — длина одной seed-строки для соответствующей части, без служебных префиксов, доменов, слов и разделителей.\n"
            "purpose — коротко укажи, в каких слотах использовать bucket. Если в сущности есть разные типы слотов, например числовой id и username, сделай разные bucket-ы. "
            "Не смешивай буквенно-цифровой seed в слоты, которые по промпту являются числовыми."
        ))
    ]
    seed_plan = seed_planner_llm.invoke(seed_plan_messages)

    seed_buckets = []
    for bucket in seed_plan.buckets:
        count = max(0, min(bucket.count, batch_size))
        if count == 0:
            continue
        length = max(3, min(bucket.length, 64))
        values = generate_seed.invoke({
            "length": length,
            "mode": bucket.mode,
            "count": count
        })
        seed_buckets.append({
            "mode": bucket.mode,
            "count": count,
            "length": length,
            "purpose": bucket.purpose,
            "values": values,
        })

    # Формируем подсказку, чтобы не повторяться (берем последние 5 примеров)
    history = list(state['unique_samples'])[-5:]
    user_content = f"Сгенерируй {batch_size} новых примеров."
    user_content += (
        f" Если помогает разнообразию, можешь использовать эти seed-bucket-ы: {seed_buckets}."
        " Это необязательные подсказки, а не требование: если заготовки мешают реалистичному формату, игнорируй их и генерируй по системному промпту."
        " Используй значения bucket-а только в слотах, подходящих его purpose: digits — для числовых частей, alnum — для username/login-like частей, letters — для буквенных частей."
        " Не вставляй alnum/letters в слот, который по системному промпту и примерам выглядит числовым."
        " Главное — корректная сущность и стиль из системного промпта."
        " В батче большинство примеров должны быть валидными и аккуратно записанными: разные допустимые разделители, регистр, префиксы или обрамление — это ок."
        " Действительно шумных вариантов с лишним пробелом, странной пунктуацией, мелкой небрежностью или похожей человеческой ошибкой делай не больше 1-2 на 10 примеров."
    )
    if history:
        user_content += f" Не повторяй эти форматы: {history}. Попробуй другие форматы записи, разделители или цифры."

    messages = [
        SystemMessage(content=sys_prompt),
        HumanMessage(content=user_content)
    ]

    # Вызов модели
    response = structured_llm.invoke(messages)


    return {
        "last_batch": response.items,
        "iterations": state['iterations'] + 1
    }


def validate_and_add_node(state: SubAgentState):
    # Просто добавляем всё, что выдала модель, в set()
    # Так как мы отказались от регулярок, тут только дедупликация
    new_items = state['last_batch']
    updated_samples = state['unique_samples'].copy()

    for item in new_items:
        if item and len(item) > 5: # Базовый фильтр от пустых строк
            updated_samples.add(item)

    return {"unique_samples": updated_samples}


def should_continue(state: SubAgentState):
    # Если набрали нужное количество или превысили лимит попыток (напр. 20)
    if len(state['unique_samples']) >= state['target_count'] or state['iterations'] > 20:
        return "end"
    return "continue"



In [19]:
# Собираем граф
workflow = StateGraph(SubAgentState)

# Добавляем узлы
workflow.add_node("generator", generate_batch_node)
workflow.add_node("validator", validate_and_add_node)

# Устанавливаем точку входа
workflow.set_entry_point("generator")

# Связываем узлы
workflow.add_edge("generator", "validator")

# Добавляем условный переход из валидатора
workflow.add_conditional_edges(
    "validator",
    should_continue,
    {
        "continue": "generator",
        "end": END
    }
)

# Компилируем
sub_agent_app = workflow.compile()

In [10]:
# input_state = {
#     'entity_key': 'PASSPORT_RF',       
#     'target_count': 100,
#     'unique_samples': set(),
#     'iterations': 0            
# }
# result = sub_agent_app.invoke(input_state,output_keys = ['entity_key', 'unique_samples'])

In [11]:
# result

In [12]:
# import json
# from datetime import datetime

# FOLDER_PATH = '../outputs/'
# entity_name = result['entity_key']
# samples = result['unique_samples']

# now = datetime.now()
# now = now.replace(microsecond=0)
# now_str = str(now).replace(' ', '_')
# print(now)
# print('../outputs/history/' + f'{entity_name}_{now_str}.json')

# total_count += len(samples)
# with open('../outputs/history/' + f'{entity_name}_{now_str}.json', 'w') as f:
#     json.dump(list(samples), f, ensure_ascii = False, indent = 4)

In [13]:
# FOLDER_PATH = '../outputs/'
# with open(FOLDER_PATH + f'{entity_name}.json', 'r') as f:
#     file = json.load(f)
#     print('Current len:', len(file))
# new = set(file)
# new.update(samples)
# print('New len:', len(new))
# with open(FOLDER_PATH + f'{entity_name}.json', 'w') as f:
#     json.dump(list(new), f, ensure_ascii = False, indent = 4)

In [20]:
import os
import json
import numpy as np
from datetime import datetime

def collect_pool(entity_key, count):
    total_cnt = 0
    total_unique_cnt = 0
    path = '../outputs/'
    for i in range(count//100):
        try:
            input_state = {
                'entity_key': entity_key,       
                'target_count': 100,
                'unique_samples': set(),
                'iterations': 0            
            }
            result = sub_agent_app.invoke(input_state,output_keys = ['entity_key', 'unique_samples'])
            
            entity_name = result['entity_key']
            samples = result['unique_samples']
            
            samples_lst = list(samples)
            
            now = datetime.now()
            now = now.replace(microsecond=0)
            now_str = str(now).replace(' ', '_')
            print(now)
            print('../outputs/history/' + f'{entity_name}_{now_str}.json')
            
            total_cnt += len(samples)
            with open('../outputs/history/' + f'{entity_name}_{now_str}.json', 'w') as f:
                json.dump(samples_lst, f, ensure_ascii = False, indent = 4)

            for i, val in enumerate(samples_lst):
                print(f'index: {i}, value : {val}')
            
            delete_idx = set(map(int, input('Delete indices: ').split()))
            samples_lst_cleaned = [sample.strip() for i, sample in enumerate(samples_lst) if i not in delete_idx]
            
            if not os.path.exists(path + f'{entity_name}.json'):
                with open(path + f'{entity_name}.json', 'w') as f:
                    json.dump(samples_lst_cleaned, f, ensure_ascii = False, indent = 4)
                print('Current len:', len(samples_lst_cleaned))
                total_unique_cnt = len(samples_lst_cleaned)
                continue
            
            with open(path + f'{entity_name}.json', 'r') as f:
                file = json.load(f)
                print('Current len:', len(file))
            new = set(file)
            new.update(samples_lst_cleaned)
            total_unique_cnt = len(new)
            print('Total unique count:', total_unique_cnt)
            with open(path + f'{entity_name}.json', 'w') as f:
                json.dump(list(new), f, ensure_ascii = False, indent = 4)
            
        except Exception as e:
            print('Exception:', e)
            break
    if total_cnt > 0 and total_unique_cnt > 0:
        with open(path + f'{entity_name}.json', 'r') as f:
            file = json.load(f)
        lengths = [len(s) for s in file]
        avg_length = np.mean(lengths)
        min_length = np.min(lengths)
        max_length = np.max(lengths)
        std_length = np.std(lengths)
        print('Statistics:', 
              f'total_cnt: {total_cnt}', 
              f'total_unique_cnt: {total_unique_cnt}', 
              f'avg_length: {avg_length}', 
              f'min_length: {min_length}', 
              f'max_length: {max_length}', 
              f'std_length: {std_length}',
              sep = '\n') 
                

In [43]:
collect_pool(entity_key = 'VK', count = 200)

2026-04-28 00:32:16
entities_pool/history/VK_2026-04-28_00:32:16.json
index: 0, value : [id86712845|Анна]
index: 1, value : @m7mhbbf278
index: 2, value : vk.com/id18163141
index: 3, value : vk.com/txtd0m6tepwn
index: 4, value : https://vk.com/doans7kaw4lh
index: 5, value : m.vk.com/id01597837
index: 6, value : vk.me/write-29968867
index: 7, value : @l83zuzgg60p6
index: 8, value : vk.com/im?sel=74334598
index: 9, value : https://m.vk.com/id94625262
index: 10, value : m.vk.com/id23414434
index: 11, value : https://m.vk.com/id69627473
index: 12, value : vk.me/pqeyeeurs32i
index: 13, value : @user_29968867
index: 14, value : https://vk.com/5cwbqqnde9rh
index: 15, value : https://vk.com/write-91909806
index: 16, value : vk.me/kb9p2cjq3967
index: 17, value : [id19927118|Алексей]
index: 18, value : vk.com/im?sel=48601533
index: 19, value : https://vk.com/ris0cduu36
index: 20, value : @jwkr71kqzie3
index: 21, value : @ij1uwg1boclx
index: 22, value : @q26aw0hwuh
index: 23, value : https://vk.co

Delete indices:  86


Current len: 889
Total unique count: 988
2026-04-28 00:34:23
entities_pool/history/VK_2026-04-28_00:34:23.json
index: 0, value : vk.com/im?sel=68114548
index: 1, value : vk.com/id67628083
index: 2, value : m.vk.com/id04369494
index: 3, value : [id70191188|Алексей]
index: 4, value : https://vk.com/163be4n5ie8g
index: 5, value : rhgc8f32hljb@vk.com
index: 6, value : m.vk.com/id79194752
index: 7, value : vk.me/id86312716?ref=chat
index: 8, value : vk.me/main3h00dz
index: 9, value : vk.me/write-18723072
index: 10, value : @i2uqgyuhb9p4 
index: 11, value : vk.com/id99353686
index: 12, value : [id19559080|Алексей]
index: 13, value : https://vk.com/write-28995504
index: 14, value : vk.com/id58054961
index: 15, value : vk.me/id43103754
index: 16, value : vk.me/write-29827059
index: 17, value : https://vk.com/okmi8datmilp
index: 18, value : vk.me/write-45410053
index: 19, value : vk.com/im?sel=82256616
index: 20, value : @cnyja2mm79
index: 21, value : https://vk.com/write-16380051
index: 22, va

Delete indices:  5 96


Current len: 988
Total unique count: 1086
Statistics:
total_cnt: 200
total_unique_cnt: 1086
avg_length: 20.028545119705342
min_length: 6
max_length: 37
std_length: 5.315819116007921


In [31]:
# import os
# import json
# FOLDER_PATH = '../outputs/'
# categories = ['PASSPORT_RF', 'BANK_CARD', 'PHONE_NUMBER', 'EMAIL', 'TELEGRAM', 'VK']
# for cat in categories:
#     input_state = {
#     'entity_key': cat,       
#     'target_count': 10,
#     'unique_samples': set(),
#     'iterations': 0            
#     }
#     res = sub_agent_app.invoke(input_state,output_keys = ['unique_samples'])['unique_samples']
#     print(cat)
#     print(res)
#     with open(FOLDER_PATH + f'{cat}.json', 'w') as f:
#         json.dump(list(res), f, ensure_ascii = False, indent = 4)
#     break
        
    

PASSPORT_RF
{'паспорт сер. 4511 № 123456', '4501 112233', '4612 889900', '4511123456', '4002-987654', '4005 654321', '0118-554433', '7715 001234', '6010-776655', '2014 334455'}
